# Open counter-UAS tracker — Colab demo

Detection and tracking research only. This notebook does not jam, spoof, intercept or otherwise engage an aircraft.

It downloads DUT Anti-UAV `video03` (about 2 MB, Apache-2.0 mirror, not stored in git), finds the dark aircraft against the sky, tracks it, and prints precision, recall and mAP@0.5.

**Runtime → Run all.** CPU is enough. No API key.

Leave `RUN_RFDETR` and `RUN_FINETUNE` set to `False` unless you want the optional learned-detector cells. Those need `requirements-dl.txt` and, for the fine-tune, a free T4.


In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

# Feature branch first so this notebook runs before the PR is merged.
# After merge, development also contains the package.
BRANCHES = ["cursor/drone-detection-skeleton-cf13", "development"]


def package_ready(root: Path) -> bool:
    return (root / "src" / "cuas" / "__init__.py").is_file()


here = Path.cwd().resolve()
root = None
for candidate in [here, *here.parents]:
    if package_ready(candidate):
        root = candidate
        break

if root is None:
    parent = Path("/content") if Path("/content").is_dir() else here
    dest = parent / "open-cuas-tracker"
    if dest.exists():
        shutil.rmtree(dest)
    cloned = False
    for branch in BRANCHES:
        try:
            subprocess.check_call(
                [
                    "git",
                    "clone",
                    "--depth",
                    "1",
                    "--branch",
                    branch,
                    "https://github.com/graboosky/open-cuas-tracker.git",
                    str(dest),
                ]
            )
        except subprocess.CalledProcessError:
            if dest.exists():
                shutil.rmtree(dest)
            continue
        if package_ready(dest):
            cloned = True
            break
        shutil.rmtree(dest)
    if not cloned:
        raise SystemExit("Could not clone a branch that contains src/cuas.")
    root = dest

os.chdir(root)
src = str(root / "src")
if src not in sys.path:
    sys.path.insert(0, src)

missing = False
for module_name in ("cv2", "pyarrow", "huggingface_hub"):
    try:
        __import__(module_name)
    except ImportError:
        missing = True
if missing or os.environ.get("CUAS_FORCE_PIP") == "1":
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(root / "requirements.txt")]
    )

print("repo", root)


## Run the clip

`contrast` is the weight-free detector measured in the README. Green boxes in the GIF are DUT annotations. Coloured boxes are tracks.


In [ ]:
from pathlib import Path

from cuas.data.dut import load_tracking_clip
from cuas.detectors import build_detector
from cuas.pipeline import run_pipeline, save_trajectory
from cuas.tracker import IoUTracker
from cuas.video_io import write_gif, write_mp4

clip = load_tracking_clip("video03")
result = run_pipeline(
    clip.frames_rgb,
    clip.ground_truth,
    build_detector("contrast"),
    IoUTracker(),
)

out = Path("runs/notebook")
out.mkdir(parents=True, exist_ok=True)
gif_path = out / "preview.gif"
write_gif(result.annotated_frames, gif_path, fps=8, max_width=480)
write_mp4(result.annotated_frames, out / "annotated.mp4", fps=10)
traj_path = out / "trajectory.png"
save_trajectory(result, clip.ground_truth, traj_path, "DUT Anti-UAV video03 / contrast")

scores = result.scores
print(f"frames: {result.n_frames}  size: {clip.width}x{clip.height}")
print(f"precision@0.5: {scores.precision:.3f}")
print(f"recall@0.5:    {scores.recall:.3f}")
print(f"mAP@0.5:       {scores.map50:.3f}")
print(f"TP {scores.true_positives}  FP {scores.false_positives}  FN {scores.false_negatives}")
print(f"detector FPS:   {result.detector_fps:.1f}")
print(f"end-to-end FPS: {result.end_to_end_fps:.1f}")
print("GIF playback is 8 fps. DUT does not state the camera frame rate.")
print("These thresholds were chosen on video03. They are not a held-out benchmark.")


In [ ]:
from IPython.display import Image, display

display(Image(filename="runs/notebook/preview.gif"))
display(Image(filename="runs/notebook/trajectory.png"))


## Optional: RF-DETR Nano (Apache-2.0)

COCO weights have no drone class, so this cell does not run during Run all. Set `RUN_RFDETR = True` on a T4 if you want a measured FPS and recall for the pretrained checkpoint. Expect weak drone recall until the fine-tune below.

Sizes `atto`, `femto`, `pico`, `xl` and `2xl` are refused. They are Platform Model License 1.0, not Apache-2.0. Do not install `rfdetr[plus]`.


In [ ]:
RUN_RFDETR = False
RFDETR_FRAMES = 8

if RUN_RFDETR:
    import subprocess
    import sys

    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "rfdetr"])
    from cuas.detectors import build_detector
    from cuas.pipeline import run_pipeline
    from cuas.tracker import IoUTracker

    learned = build_detector("rfdetr", size="nano")
    learned_result = run_pipeline(
        clip.frames_rgb[:RFDETR_FRAMES],
        clip.ground_truth[:RFDETR_FRAMES],
        learned,
        IoUTracker(),
    )
    learned_scores = learned_result.scores
    print(f"RF-DETR Nano on {learned_result.n_frames} frames of video03")
    print(f"precision@0.5: {learned_scores.precision:.3f}")
    print(f"recall@0.5:    {learned_scores.recall:.3f}")
    print(f"mAP@0.5:       {learned_scores.map50:.3f}")
    print(f"detector FPS:   {learned_result.detector_fps:.1f}")
    print(f"end-to-end FPS: {learned_result.end_to_end_fps:.1f}")
    print("COCO class names are not 'uav'. Low recall here means the checkpoint was not fine-tuned.")
else:
    print("RF-DETR skipped. Metrics for this cell: TBD – run notebook with RUN_RFDETR = True.")


## Optional fine-tune on a small DUT subset

Stays off for Run all. On a free T4, set `RUN_FINETUNE = True`. The cell exports at most 100 detection images from one public parquet shard (about 124 MB) into a COCO tree and trains RF-DETR Nano for 3 epochs. Checkpoints go to Google Drive when Colab is available, otherwise to `runs/rfdetr-nano` (gitignored). They are not committed.

This cell was not executed for the numbers in the README.


In [ ]:
RUN_FINETUNE = False
MAX_IMAGES = 100
EPOCHS = 3

if RUN_FINETUNE:
    import subprocess
    import sys
    from pathlib import Path

    import pyarrow.parquet as pq
    from huggingface_hub import hf_hub_download

    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "rfdetr[train]"]
    )
    shard = hf_hub_download(
        repo_id="aegean-ai/dut-anti-uav",
        filename="detection-parquet/test/test-00000.parquet",
        repo_type="dataset",
    )
    from cuas.data.coco import export_coco_tree, rows_from_detection_table

    rows = rows_from_detection_table(pq.read_table(shard))[:MAX_IMAGES]
    try:
        from google.colab import drive
    except ImportError:
        drive = None
    if drive is not None:
        drive.mount("/content/drive")
        work = Path("/content/drive/MyDrive/open-cuas-tracker")
    else:
        work = Path("runs")
    dataset_dir = export_coco_tree(rows, work / "dut-subset")
    output_dir = work / "rfdetr-nano"
    output_dir.mkdir(parents=True, exist_ok=True)

    from rfdetr import RFDETRNano

    model = RFDETRNano()
    model.train(
        dataset_dir=str(dataset_dir),
        epochs=EPOCHS,
        batch_size=2,
        grad_accum_steps=4,
        lr=1e-4,
        output_dir=str(output_dir),
    )
    print("weights under", output_dir)
    print("Load them later with build_detector('rfdetr', size='nano', weights='<checkpoint>').")
else:
    print("Fine-tune skipped. Weights are not produced by Run all.")
